In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "workspace")
catalogo = dbutils.widgets.get("catalogo")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.gold")

# Gera uma chave substituta BIGINT a partir de uma ou mais colunas.
# sha2 transforma o valor num código hexadecimal fixo (o mesmo valor gera sempre o mesmo código).
# Os 15 primeiros caracteres desse código são convertidos de hexadecimal (base 16) para número (base 10),
# o que cabe num BIGINT, como o enunciado pede.
def gerar_sk(*colunas):
    texto = F.concat_ws("||", *[F.col(c) for c in colunas])
    return F.conv(F.substring(F.sha2(texto, 256), 1, 15), 16, 10).cast("bigint")

In [0]:
filmes = spark.table(f"{catalogo}.silver.tb_info_filmes")

# Uma linha por filme, com a chave substituta e os dados descritivos.
# Entram todos os filmes, inclusive os não lançados; o filtro de "Lançado" é feito na fato.
dim_movies = filmes.select(
    gerar_sk("id_filme").alias("sk_movie_id"),
    "id_filme", "titulo", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse",
)

dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.gold.dim_movies")
print(dim_movies.count())

97611


In [0]:
dim_movies = spark.table(f"{catalogo}.gold.dim_movies")
financeiro = spark.table(f"{catalogo}.silver.tb_financeiro_filmes")
metricas = spark.table(f"{catalogo}.silver.tb_metricas_engajamento")

# Grão: um filme lançado por linha.
# Financeiro e métricas já têm uma linha por filme na Silver, então os joins não duplicam linhas.
# Left join: um filme lançado sem dado financeiro continua na fato, com NULL nessas colunas.
fato = (
    dim_movies.filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
    .join(financeiro, "id_filme", "left")
    .join(metricas, "id_filme", "left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

fato.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.gold.fact_movies_performance")

# Confere o grão: o número de linhas tem que ser igual ao número de filmes distintos
print("linhas:", fato.count())
print("filmes distintos:", fato.select("sk_movie_id").distinct().count())

linhas: 96261
filmes distintos: 96261


In [0]:
dim_movies = spark.table(f"{catalogo}.gold.dim_movies").select("sk_movie_id", "id_filme")
generos = spark.table(f"{catalogo}.silver.tb_generos")

# dim_genres: catálogo único de gêneros, com a chave substituta
dim_genres = (
    generos.select("nome_genero").distinct()
    .withColumn("sk_genre_id", gerar_sk("nome_genero"))
    .select("sk_genre_id", "nome_genero")
)
dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.gold.dim_genres")

# bridge_movie_genre: um par (filme, gênero) por linha.
# O join com dim_movies traz o sk_movie_id; o sk_genre_id é calculado pelo mesmo hash da dimensão,
# então o número bate sem precisar fazer join com dim_genres.
bridge_movie_genre = (
    generos.join(dim_movies, "id_filme")
    .withColumn("sk_genre_id", gerar_sk("nome_genero"))
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)
bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.gold.bridge_movie_genre")

print("dim_genres:", dim_genres.count())
print("bridge_movie_genre:", bridge_movie_genre.count())

dim_genres: 19
bridge_movie_genre: 140078


In [0]:
entidades = spark.table(f"{catalogo}.silver.tb_pessoas_empresas")

# Separa pessoas físicas de produtoras: cada grupo vai para uma dimensão diferente
pessoas = entidades.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
produtoras = entidades.filter(F.col("tipo_entidade") == "Produtora")

# dim_people: a chave é nome + tipo. Quem atua e dirige aparece duas vezes, uma por papel.
dim_people = (
    pessoas.select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .distinct()
    .withColumn("sk_person_id", gerar_sk("nome_pessoa", "tipo_pessoa"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)
dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.gold.dim_people")

bridge_movie_person = (
    pessoas.join(dim_movies, "id_filme")
    .withColumn("sk_person_id", gerar_sk("nome_entidade", "tipo_entidade"))
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)
bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.gold.bridge_movie_person")

# dim_companies: catálogo único de produtoras
dim_companies = (
    produtoras.select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", gerar_sk("nome_produtora"))
    .select("sk_company_id", "nome_produtora")
)
dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.gold.dim_companies")

bridge_movie_company = (
    produtoras.join(dim_movies, "id_filme")
    .withColumn("sk_company_id", gerar_sk("nome_entidade"))
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)
bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.gold.bridge_movie_company")

print("dim_people:", dim_people.count())
print("bridge_movie_person:", bridge_movie_person.count())
print("dim_companies:", dim_companies.count())
print("bridge_movie_company:", bridge_movie_company.count())

dim_people: 417527
bridge_movie_person: 754365
dim_companies: 44431
bridge_movie_company: 114635


In [0]:
avaliacoes = spark.table(f"{catalogo}.silver.tb_avaliacoes_usuarios")

# Resume as avaliações por filme: quantidade e média das notas (2 casas).
# A média ignora notas NULL (as que estavam fora de 0 a 10).
# Só entram filmes que existem em dim_movies, senão o sk_movie_id ficaria sem correspondência.
dim_reviews = (
    avaliacoes.join(dim_movies, "id_filme")
    .groupBy("id_filme", "sk_movie_id")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios"),
    )
    # Prefixo "review_" para a chave não ficar igual à sk_movie_id do mesmo filme
    .withColumn("chave_review", F.concat(F.lit("review_"), F.col("id_filme")))
    .withColumn("sk_review_id", gerar_sk("chave_review"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.gold.dim_reviews")

print("dim_reviews:", dim_reviews.count())

dim_reviews: 27226


In [0]:
data_limite = spark.sql(f"""
    SELECT MAX(data_lancamento) AS data_limite
    FROM {catalogo}.gold.dim_movies
    WHERE status_filme = 'Lançado'
      AND data_lancamento <= current_date()
""").first()["data_limite"]

print("Data limite:", data_limite)

Data limite: 2026-02-19


In [0]:
#1
display(spark.sql(f"""
    SELECT SUM(receita_brl) AS receita_total_brl
    FROM {catalogo}.gold.fact_movies_performance
"""))

receita_total_brl
806537380275.18


In [0]:
#2
display(spark.sql(f"""
    SELECT m.titulo, f.popularidade
    FROM {catalogo}.gold.fact_movies_performance f
    JOIN {catalogo}.gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


In [0]:
#3
display(spark.sql(f"""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM {catalogo}.gold.bridge_movie_genre b
    JOIN {catalogo}.gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))

nome_genero,qtd_filmes
Drama,32196
Documentary,18943
Comedy,18572
Thriller,10255
Horror,9704
Romance,7625
Action,6039
Crime,4731
Animation,4459
TV Movie,4073


In [0]:
#4
display(spark.sql(f"""
    WITH ranking AS (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao
        FROM {catalogo}.gold.fact_movies_performance f
        JOIN {catalogo}.gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    SELECT posicao, titulo, receita_usd, receita_brl
    FROM ranking
    WHERE posicao <= 10
    ORDER BY posicao
"""))

posicao,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,13913760000.00
2,Avatar: The Way of Water,2320250281.00,11529787696.35
3,AVENGERS: INFINITY WAR,2052415039.00,10198860811.80
4,spider-man: no way home,1921847111.00,9550042663.98
5,The Lion King,1663075401.00,8264154282.65
6,Top Gun: Maverick,1488732821.00,7397811134.11
7,Barbie,1428545028.00,7098725953.14
8,The Super Mario Bros. Movie,1355725263.00,6736869976.90
9,Black Panther,1349926083.00,6708052691.64
10,Star Wars: The Last Jedi,1332698830.00,6622447026.04


In [0]:
#5
display(spark.sql(f"""
    WITH participacoes AS (
        SELECT p.nome_pessoa, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
        FROM {catalogo}.gold.bridge_movie_person b
        JOIN {catalogo}.gold.dim_people p ON p.sk_person_id = b.sk_person_id
        JOIN {catalogo}.gold.dim_movies m ON m.sk_movie_id = b.sk_movie_id
        WHERE p.tipo_pessoa = 'Ator'
          AND m.status_filme = 'Lançado'
          AND m.data_lancamento >  add_months(DATE'{data_limite}', -24)
          AND m.data_lancamento <= DATE'{data_limite}'
        GROUP BY p.nome_pessoa
    )
    SELECT RANK() OVER (ORDER BY qtd_filmes DESC) AS posicao, nome_pessoa AS ator, qtd_filmes
    FROM participacoes
    ORDER BY posicao
    LIMIT 10
"""))

posicao,ator,qtd_filmes
1,Kevin Hart,64
2,Alex Dauphin,59
2,Milana Vayntrub,59
2,Forrest Conoly,59
2,John Travolta,59
2,John Cena,59
2,David Chan,59
2,Brandon H. Morgan,59
2,Greg Kriek,59
2,David Chang,59


In [0]:
#6
display(spark.sql(f"""
    WITH lucro_produtora AS (
        SELECT c.nome_produtora,
               SUM(f.lucro_usd) AS lucro_total_usd,
               SUM(f.lucro_brl) AS lucro_total_brl,
               COUNT(DISTINCT f.sk_movie_id) AS qtd_filmes
        FROM {catalogo}.gold.bridge_movie_company b
        JOIN {catalogo}.gold.dim_companies c ON c.sk_company_id = b.sk_company_id
        JOIN {catalogo}.gold.dim_movies m ON m.sk_movie_id = b.sk_movie_id
        JOIN {catalogo}.gold.fact_movies_performance f ON f.sk_movie_id = b.sk_movie_id
        WHERE f.lucro_usd IS NOT NULL
          AND m.data_lancamento >  add_months(DATE'{data_limite}', -60)
          AND m.data_lancamento <= DATE'{data_limite}'
        GROUP BY c.nome_produtora
    )
    SELECT RANK() OVER (ORDER BY lucro_total_usd DESC) AS posicao, nome_produtora,
           lucro_total_usd, lucro_total_brl, qtd_filmes
    FROM lucro_produtora
    ORDER BY posicao
    LIMIT 10
"""))

posicao,nome_produtora,lucro_total_usd,lucro_total_brl,qtd_filmes
1,Universal Pictures,5772329679.00,28683860640.89,24
2,Marvel Studios,4953462823.00,24614747460.07,8
3,Columbia Pictures,3662050755.00,18197462611.75,12
4,Pascal Pictures,2701952454.00,13426542134.42,3
5,Illumination,2431353473.00,12081881678.03,3
6,Paramount,2239394101.00,11127997166.70,14
7,Kevin Feige Productions,2138205367.00,10625170109.71,4
8,20th Century Studios,2074815383.00,10310172601.21,8
9,Lightstorm Entertainment,1860250281.00,9243955696.35,1
10,Heyday Films,1490495872.00,7406572087.14,2
